# RecSys 2026 — LoRA Fine-tune for Blind-A (Colab)

Runs the full LoRA fine-tune on Colab GPU, then zips the trained adapter so you can download it.

## Setup checklist

1. **Runtime → Change runtime type → A100 GPU** (best). T4/L4 also work, just slower.
2. Run all cells.
3. When training finishes, the adapter zip will be at `/content/qwen3b_blinda_v1.zip`. Download it via the file browser (📁 sidebar) **OR** mount Drive in the optional last cell to auto-save it.

Total wall time on A100: ~50 min. Adapter zip is ~80 MB.

**Drive is NOT required** — the repo is cloned from GitHub; only mount Drive if you want the adapter to survive Colab disconnects.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone the repo (public). Replace with the SSH URL if you forked privately.
!git clone https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

In [ ]:
# 3) Install pinned deps. Colab usually has compatible torch — pip resolves the rest.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, trl, peft, bm25s; print('torch', torch.__version__, 'transformers', transformers.__version__, 'trl', trl.__version__, 'peft', peft.__version__)"

In [ ]:
# 4) (Optional) HuggingFace token for faster downloads.
# import os; os.environ['HF_TOKEN'] = 'hf_...'  # uncomment + paste your token

In [ ]:
# 5) Run the full training pipeline. We skip:
#    - SKIP_VENV: Colab's Python is fine, no need for a venv.
#    - SKIP_INFERENCE / SKIP_ZIP: those run on your local M4 with the downloaded adapter
#      (much cheaper than a Colab session for the 80-row inference + zip).
# Bump batch + max_length to A100-friendly defaults.
!SKIP_VENV=1 \
  SKIP_INFERENCE=1 \
  SKIP_ZIP=1 \
  PER_DEVICE_BATCH=2 \
  GRAD_ACCUM=8 \
  TRAIN_MAX_LENGTH=3072 \
  bash run_pipeline.sh

In [ ]:
# 6) Zip the trained adapter into /content/ so you can download it.
import shutil, os
ADAPTER_DIR = 'lora_adapters/qwen3b_blinda_v1/final_adapter'
ZIP_BASE = '/content/qwen3b_blinda_v1'
assert os.path.isdir(ADAPTER_DIR), f'adapter not found at {ADAPTER_DIR} — did training fail?'
shutil.make_archive(ZIP_BASE, 'zip', ADAPTER_DIR)
print('wrote', ZIP_BASE + '.zip')
!ls -lh {ZIP_BASE}.zip

In [ ]:
# 7a) Download via browser (no Drive needed).
from google.colab import files
files.download('/content/qwen3b_blinda_v1.zip')

In [ ]:
# 7b) OPTIONAL — mount Drive ONLY if you want the adapter to survive Colab disconnects.
# Uncomment the lines below to save a copy to Drive.
#
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# os.makedirs('/content/drive/MyDrive/recsys2026-adapters', exist_ok=True)
# shutil.copy('/content/qwen3b_blinda_v1.zip', '/content/drive/MyDrive/recsys2026-adapters/')
# print('saved to Drive')

## Done. On your local M4 / Linux:

```bash
cd recsys2026-lora-tutorial
mkdir -p lora_adapters/qwen3b_blinda_v1/final_adapter
unzip ~/Downloads/qwen3b_blinda_v1.zip -d lora_adapters/qwen3b_blinda_v1/final_adapter

# Inference + package only (training was done on Colab).
SKIP_DATASET=1 SKIP_TRAIN=1 ./run_pipeline.sh
```

`output/prediction.zip` is ready to upload to CodaBench.